# Vectores Semánticos de juguete 🧸
**Idea del paper (Word2Vec):** cada palabra se convierte en una lista de números (un **vector**).

- Palabras parecidas → vectores **cerca**
- Palabras distintas → vectores **lejos**

¿Y de dónde salen los números? De las **frases**: *"dime con quién andas y te diré quién eres"*.

El paper propone **dos algoritmos** para que la computadora aprenda esos vectores sola leyendo texto:
- **CBOW:** ve las palabras vecinas y adivina la del centro → *"el ___ usa corona"* → **rey**
- **Skip-gram:** ve una palabra y adivina sus vecinas → **rey** → *"usa", "corona"*

Aquí hacemos una versión simplificada: **contamos con qué palabras aparece cada palabra**.

In [1]:
import numpy as np

## 1. De frases a vectores
Para cada palabra, contamos en cuántas frases aparece junto a cada **palabra de contexto**.
Ese conteo es su vector.

In [2]:
def crear_vector(palabra, frases, contexto):
    vector = []
    for c in contexto:
        veces = 0
        for frase in frases:
            if palabra in frase.split() and c in frase.split():
                veces += 1
        vector.append(veces)
    return np.array(vector)

## 2. Ejemplo 1: Relación semántica (de significado)
Estas son las frases que "lee" la computadora:

In [3]:
frases1 = [
    "el rey usa corona",
    "la reina usa corona",
    "el hombre camina",
    "la mujer camina",
    "un perro ladra",
]

contexto1 = ["corona", "el", "la", "ladra"]

palabras = {}
for p in ["rey", "reina", "hombre", "mujer", "perro"]:
    palabras[p] = crear_vector(p, frases1, contexto1)

print("columnas:", contexto1)
for palabra, vector in palabras.items():
    print(f"{palabra:7s}", vector)

columnas: ['corona', 'el', 'la', 'ladra']
rey     [1 1 0 0]
reina   [1 0 1 0]
hombre  [0 1 0 0]
mujer   [0 0 1 0]
perro   [0 0 0 1]


Cómo leer la matriz:
- **rey** aparece con *corona* y con *el* → `[1, 1, 0, 0]`
- **reina** aparece con *corona* y con *la* → `[1, 0, 1, 0]`
- **perro** solo aparece con *ladra* → `[0, 0, 0, 1]`

Nadie le dijo a la computadora qué es "rey": lo sabe por **con quién anda**.

## 3. ¿Qué tan cerca están dos palabras? → Distancia euclidiana
Es la distancia "en línea recta" de toda la vida (Pitágoras):
1. Restamos los vectores
2. Elevamos al cuadrado
3. Sumamos
4. Sacamos raíz

**Distancia chica = se parecen. Distancia grande = no se parecen.**

In [4]:
def distancia(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

print("rey vs hombre:", round(distancia(palabras["rey"], palabras["hombre"]), 2))
print("rey vs reina :", round(distancia(palabras["rey"], palabras["reina"]), 2))
print("rey vs perro :", round(distancia(palabras["rey"], palabras["perro"]), 2))

rey vs hombre: 1.0
rey vs reina : 1.41
rey vs perro : 1.73


## 4. Matemáticas con palabras
Lo más famoso del paper: **rey − hombre + mujer = ?**

In [5]:
resultado = palabras["rey"] - palabras["hombre"] + palabras["mujer"]
print("rey     ", palabras["rey"])
print("- hombre", palabras["hombre"])
print("+ mujer ", palabras["mujer"])
print("=       ", resultado, "\n")

for palabra, vector in palabras.items():
    print(palabra, "->", round(distancia(resultado, vector), 2))
# La distancia más chica es 'reina' (0.0) 👑

rey      [1 1 0 0]
- hombre [0 1 0 0]
+ mujer  [0 0 1 0]
=        [1 0 1 0] 

rey -> 1.41
reina -> 0.0
hombre -> 1.73
mujer -> 1.0
perro -> 1.73


## 5. Ejemplo 2: Relación sintáctica (de forma de la palabra)
El paper también pregunta: *¿qué palabra es a "pequeño" lo que "grandísimo" es a "grande"?*

Frases nuevas:

In [6]:
frases2 = [
    "el elefante es grande",
    "wow el elefante es grandísimo",
    "el ratón es pequeño",
    "wow el ratón es pequeñísimo",
]

contexto2 = ["elefante", "ratón", "wow"]

tamaños = {}
for p in ["grande", "grandísimo", "pequeño", "pequeñísimo"]:
    tamaños[p] = crear_vector(p, frases2, contexto2)

print("columnas:", contexto2)
for palabra, vector in tamaños.items():
    print(f"{palabra:12s}", vector)

columnas: ['elefante', 'ratón', 'wow']
grande       [1 0 0]
grandísimo   [1 0 1]
pequeño      [0 1 0]
pequeñísimo  [0 1 1]


Cómo leer la matriz:
- **grande** y **grandísimo** aparecen con *elefante* (cosas grandes)
- **pequeño** y **pequeñísimo** aparecen con *ratón* (cosas pequeñas)
- Las palabras con **-ísimo** aparecen con *wow* (exageración)

**grandísimo − grande + pequeño = ?**

In [7]:
resultado2 = tamaños["grandísimo"] - tamaños["grande"] + tamaños["pequeño"]
print("grandísimo", tamaños["grandísimo"])
print("- grande  ", tamaños["grande"])
print("+ pequeño ", tamaños["pequeño"])
print("=         ", resultado2, "\n")

for palabra, vector in tamaños.items():
    print(palabra, "->", round(distancia(resultado2, vector), 2))
# La distancia más chica es 'pequeñísimo' (0.0) 🐜

grandísimo [1 0 1]
- grande   [1 0 0]
+ pequeño  [0 1 0]
=          [0 1 1] 

grande -> 1.73
grandísimo -> 1.41
pequeño -> 1.0
pequeñísimo -> 0.0


## Resumen
1. Cada palabra es un **vector**, y sus números salen de **con qué palabras aparece** en las frases.
2. Con la **distancia** sabemos qué palabras se parecen.
3. Podemos **sumar y restar** palabras:
   - Semántica: rey − hombre + mujer = reina
   - Sintáctica: grandísimo − grande + pequeño = pequeñísimo
4. En el paper real, la computadora aprende los vectores con **CBOW** o **Skip-gram**, leyendo millones de frases.